# AgriSmart soil image classifier training

Run this notebook in Google Colab with a GPU runtime. It uploads and runs `backend/soil_model_training.py`, which downloads the pinned **original** dataset, audits its labels, makes stratified train/validation/test splits, fine-tunes MobileNetV2, and reports actual held-out metrics.

The source dataset documents seven labels: **Alluvial, Arid, Black, Laterite, Mountain, Red, and Yellow Soil**. It does not contain Sandy, Clay, or Loamy classes; this training pipeline does not invent or map those classes. Dataset source: [Soil Classification Dataset](https://github.com/Phantom-fs/Soil-Classification-Dataset), paper [doi:10.1016/j.engappai.2025.111425](https://doi.org/10.1016/j.engappai.2025.111425), license **CC BY-NC 4.0**. Use only for non-commercial research/education and keep attribution. The original images are used; CyAUG synthetic images are excluded from evaluation.

This is a closed-set classifier, not an NPK/pH test or a guaranteed non-soil detector. Do not treat dataset test accuracy as expected field-photo accuracy.

## 1. Select the runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). The notebook prints the actual device/runtime; it does not claim a metric until evaluation completes.

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Visible GPU devices:', tf.config.list_physical_devices('GPU'))

## 2. Upload the training script

From the AgriSmart project, upload `backend/soil_model_training.py` when prompted. The script pins the dataset commit, checks the exact seven class folders, removes exact duplicates, and fails if the source labels unexpectedly change.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
script_name = next((name for name in uploaded if name.endswith('soil_model_training.py')), None)
if script_name is None:
    raise RuntimeError('Upload backend/soil_model_training.py from the AgriSmart project.')
Path('/content/soil_model_training.py').write_bytes(uploaded[script_name])
print('Training script ready:', Path('/content/soil_model_training.py').stat().st_size, 'bytes')

## 3. Install the training dependencies

Colab normally includes TensorFlow; this cell ensures the script's remaining packages are available.

In [ ]:
%pip install -q 'tensorflow>=2.15' 'scikit-learn>=1.3' 'matplotlib>=3.7' 'pillow>=10'

## 4. Download, train, fine-tune, and evaluate

The default run uses 70/15/15 stratified splits, ImageNet MobileNetV2, training-only augmentation, a frozen-head phase, and fine-tuning. The script prints held-out accuracy/F1 and per-class precision/recall/F1 and writes the confusion matrix and accuracy/loss plots from this run. No metrics are prefilled.

In [ ]:
!python /content/soil_model_training.py --output-dir /content/soil_classifier_export

## 5. Download and install the model bundle

The ZIP contains `soil_classifier.keras`, `soil_class_names.json`, `soil_classifier_metadata.json`, and the evaluation plots. Extract the three model/JSON files directly into the AgriSmart project's `models/` folder. Keep the metadata paired with the exact model and class list.

In [ ]:
from google.colab import files
files.download('/content/soil_classifier_export/AgriSmart_Soil_Classifier.zip')